# Notebook 01 — NGS Technologies and Applications

**Module:** 09 — Genomics and NGS  
**Notebook:** 01 of 16  
**Time estimate:** 60 minutes

> **Track A requirement:** Be able to compare Illumina, PacBio, and ONT in
> one sentence each, and name the appropriate technology for each application.

---
## Step 1 — Motivation

Every downstream analysis in this module starts with a FASTQ file produced by a
sequencing machine. Understanding how that file was generated — and what errors or
biases it carries — is prerequisite knowledge for every QC, alignment, and variant
calling decision you will make.

---
## Step 2 — Intuition

Three generations of sequencing technology:

**Illumina (short reads):** Sequence by synthesis — add one labelled nucleotide at a
time, photograph the colour, repeat. Reads are 50–300 bp. Very high accuracy (Q30+)
and throughput. Cannot span repeats longer than the read.

**PacBio SMRT (long reads):** Watch a single DNA polymerase in real time as it
incorporates fluorescent nucleotides. Reads are 10–30 kb. High error rate per base
(~1–5%) but can span repeats. HiFi mode averages multiple passes: Q20+ at 15–25 kb.

**Oxford Nanopore (long reads):** Thread a DNA strand through a protein pore;
measure current disruptions to identify bases. Reads up to 1 Mb. Raw error rate
~5%; R10 chemistry brings this to ~Q20. Real-time output.

The right technology depends on the question.

---
## Step 3 — Biological Background

**Applications and the technology that fits each:**

| Application | Technology | Why |
|------------|-----------|-----|
| WGS (large cohort) | Illumina 150 bp PE | High throughput, low cost per base |
| WGS (de novo assembly) | PacBio HiFi or ONT | Span repeats for contiguous assembly |
| RNA-seq | Illumina 75–150 bp | High depth, mature tools |
| Long-read transcriptomics | PacBio IsoSeq / ONT | Full-length isoforms without assembly |
| Targeted sequencing (panel) | Illumina | Deep coverage of known loci |
| Structural variant detection | ONT or PacBio | SVs require long reads |
| Metagenomics | ONT (rapid, real-time) | Field sequencing, fast turnaround |
| ChIP-seq / ATAC-seq | Illumina 50 bp SE or PE | Short fragments, high depth |

**The standard Illumina NGS pipeline:**
```
Sample → Library prep → Sequencing → FASTQ → QC → Alignment → BAM → Analysis
```

---
## Step 4 — Mathematical Explanation

**Coverage depth (C):**
$$C = \frac{N \times L}{G}$$

where $N$ = number of reads, $L$ = read length, $G$ = genome size.

**Coverage uniformity:** Poisson model — if $C$ is the mean depth, the probability
of a position having depth $k$ is:
$$P(depth = k) = \frac{e^{-C} C^k}{k!}$$

At 30× coverage, the probability of zero coverage at any given position is:
$P(k=0) = e^{-30} \approx 9 \times 10^{-14}$ (negligible). But in practice, coverage
is far from Poisson due to GC bias and library complexity.

In [ ]:
# Step 6 — Python: coverage calculations and technology comparison
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
import math


def coverage_depth(n_reads: int, read_length: int, genome_size: int) -> float:
    return (n_reads * read_length) / genome_size


def reads_needed_for_coverage(target_coverage: float, read_length: int, genome_size: int) -> int:
    return math.ceil(target_coverage * genome_size / read_length)


def cost_estimate_illumina_per_gb(year: int = 2024) -> float:
    # NIH cost-per-genome trend (approximate)
    return 10.0  # ~$10/Gb as of 2024


# Coverage examples
examples = [
    ('Human WGS 30×', 900_000_000, 150, 3_100_000_000),
    ('Mouse WGS 30×', 540_000_000, 150, 2_700_000_000),
    ('RNA-seq 50M reads', 50_000_000, 75, 3_100_000_000),
    ('Targeted panel 500×', 1_000_000, 150, 300_000),
]

print(f"{'Application':<30} {'Reads':>15} {'Read L':>8} {'Coverage':>10}")
print('-' * 65)
for app, n, l, g in examples:
    cov = coverage_depth(n, l, g)
    print(f"{app:<30} {n:>15,} {l:>8} {cov:>9.1f}×")

print()
# How many reads to reach 30× of human genome at different read lengths?
print("Reads needed for 30× human genome (3.1 Gb):")
for read_len in [50, 75, 100, 150, 250]:
    n = reads_needed_for_coverage(30, read_len, 3_100_000_000)
    gb = n * read_len / 1e9
    print(f"  {read_len} bp reads: {n:,} reads ({gb:.1f} Gb)")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.stats import poisson

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, fig)

# Panel A: Coverage depth distribution under Poisson model
ax1 = fig.add_subplot(gs[0, 0])
k_vals = np.arange(0, 80)
for mean_cov, color in [(10, 'blue'), (30, 'green'), (50, 'red')]:
    probs = poisson.pmf(k_vals, mean_cov)
    ax1.plot(k_vals, probs, color=color, label=f'{mean_cov}× mean coverage')
ax1.set_xlabel('Coverage depth at a position')
ax1.set_ylabel('Probability (Poisson model)')
ax1.set_title('A. Poisson coverage model\n(assumes uniform library)')
ax1.legend(fontsize=8)

# Panel B: Technology comparison radar-like bar chart
ax2 = fig.add_subplot(gs[0, 1])
technologies = ['Illumina\nNovaseq', 'PacBio\nHiFi', 'Oxford\nNanopore']
metrics = {
    'Read length (log10 bp)': [np.log10(150), np.log10(15000), np.log10(50000)],
    'Accuracy (Q score)': [35, 25, 20],
    'Throughput (relative)': [10, 3, 5],
    'Cost per Gb (inverse, relative)': [10, 2, 4],
}
x = np.arange(len(technologies))
colors = ['steelblue', 'tomato', 'forestgreen']
ax2.bar(x, metrics['Accuracy (Q score)'], color=colors, alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(technologies, fontsize=9)
ax2.set_ylabel('Mean accuracy (Phred Q)')
ax2.set_title('B. Sequencing accuracy by technology\n(approximate, 2024)')
ax2.axhline(30, color='k', ls='--', label='Q30 threshold', alpha=0.5)
ax2.legend(fontsize=8)

# Panel C: Full NGS pipeline flowchart (text-based)
ax3 = fig.add_subplot(gs[1, 0])
ax3.axis('off')
pipeline_steps = [
    ('FASTQ', 'Raw reads from sequencer'),
    ('QC', 'FastQC; Trimmomatic/fastp'),
    ('Alignment', 'BWA-MEM (DNA) or STAR (RNA)'),
    ('BAM', 'SAM → sort → index → BAM'),
    ('Processing', 'Mark duplicates; BQSR'),
    ('Analysis', 'Variant calling OR quantification'),
    ('Output', 'VCF (DNA) or count matrix (RNA)'),
]
colors_step = ['#E3F2FD', '#E8F5E9', '#FFF3E0', '#E8F5E9', '#FFF3E0', '#E3F2FD', '#FCE4EC']
for i, (step, desc) in enumerate(pipeline_steps):
    y = 0.9 - i * 0.13
    ax3.add_patch(plt.Rectangle((0.05, y - 0.05), 0.9, 0.10,
                                  facecolor=colors_step[i], edgecolor='gray', lw=1.5))
    ax3.text(0.5, y, f'{step}: {desc}', ha='center', va='center', fontsize=9)
    if i < len(pipeline_steps) - 1:
        ax3.annotate('', xy=(0.5, y - 0.05), xytext=(0.5, y - 0.03),
                     arrowprops=dict(arrowstyle='->', color='gray'))
ax3.set_title('C. Standard NGS analysis pipeline', fontsize=10)

# Panel D: Read length vs. application
ax4 = fig.add_subplot(gs[1, 1])
applications = ['ChIP-seq', 'RNA-seq', 'WGS (SNPs)', 'WES', 'Metagenomics', 'De novo assembly', 'SV calling']
min_len = [50, 50, 100, 100, 150, 10000, 5000]
max_len = [75, 150, 150, 150, 250, 50000, 50000]
y_pos = range(len(applications))
ax4.barh(y_pos, [m - n for m, n in zip(max_len, min_len)],
         left=min_len, color='steelblue', alpha=0.7)
ax4.barh(y_pos, min_len, color='lightgray', alpha=0.5)
ax4.set_yticks(y_pos)
ax4.set_yticklabels(applications, fontsize=9)
ax4.set_xscale('log')
ax4.set_xlabel('Read length (bp, log scale)')
ax4.set_title('D. Read length requirements by application')
ax4.axvline(150, color='green', ls='--', label='Illumina max', alpha=0.7)
ax4.axvline(10000, color='orange', ls='--', label='PacBio/ONT min', alpha=0.7)
ax4.legend(fontsize=7)

plt.suptitle('Module 09: NGS Technology Landscape', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ngs_technologies.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Module 09 interview answer hierarchy

| Interview question | Notebook that prepares it |
|-------------------|---------------------------|
| Walk me through a variant calling pipeline | NB01–NB09 complete view; NB14 implementation |
| What is a MAPQ score? | NB03, NB05 |
| How do you decode a CIGAR string? | NB04 |
| What is the difference between STAR and BWA? | NB03, NB10 |
| What is TPM and why is it preferred over RPKM? | NB11 |
| How does DESeq2 normalize counts? | NB12 |
| What is duplicate marking and why does it matter? | NB06 |
| How do you read a VCF file? | NB09 |

---
## Step 8 — Exercises

See `exercises/01_ngs_orientation_exercises.md`.

1. Calculate the number of reads needed for 100× coverage of a 50 Mb exome with
   150 bp reads.
2. An RNA-seq experiment produces 40M paired-end 75 bp reads. What is the effective
   coverage of a 12 kb gene? Of the 3 Gb genome?
3. Match each application to the best technology: (a) de novo assembly of a bacterial
   genome, (b) identifying SNPs in 1000 patients, (c) detecting full-length mRNA
   isoforms in a tumour.

---
## Step 10 — Quiz

1. What does "paired-end" sequencing mean? Why is it useful for alignment?
2. A sequencing run produces 30 Gb of data at 150 bp read length. How many reads is that?
3. What type of sequencing error is most common in Illumina data? In Oxford Nanopore data?
4. Why can short reads not span structural variants reliably?
5. What is library complexity, and why does low complexity hurt coverage uniformity?

---
## Step 12 — Reflection

> *[In one paragraph: explain to a wet-lab biologist why sequencing a human genome to
> 30× means you sequence about 3× the genome's worth of data, and why that still isn't
> enough for some analyses.]*

---
*Next: `02_read_quality_control_fastqc.ipynb`*